In [2]:
import numpy as np

In [3]:
def attn(X, indices, gamma):
    X = np.asarray(X, dtype=float)
    indices = np.asarray(indices, dtype=int)
    Y = np.empty_like(X)
    
    one_minus_gamma = 1.0 - gamma
    
    # Проходим по каждому сегменту, заданному границами
    for i in range(len(indices) - 1):
        start, end = indices[i], indices[i+1]
        if start == end:
            continue
            
        segment = X[start:end]
        n = len(segment)
        
        if gamma == 0:
            Y[start:end] = segment
        else:
            # Векторизованное вычисление EMA:
            # Y[t] = (1-gamma) * gamma^t * cumsum(X[t] * gamma^-t)
            powers = gamma ** np.arange(n)
            weighted = segment / powers
            cumweighted = np.cumsum(weighted)
            Y[start:end] = one_minus_gamma * powers * cumweighted
            
    return Y

In [4]:
def attn_vectorized(X, indices, gamma):
    X = np.asarray(X, dtype=float)
    indices = np.asarray(indices, dtype=int)
    I = len(X)
    Y = np.empty_like(X)

    if gamma == 0:
        return X.copy()
    
    # Глобальные степени гаммы
    arange = np.arange(I)
    powers = gamma ** arange
    powers_inv = 1.0 / powers  # gamma ** -arange
    
    # Взвешенные значения для кумулятивной суммы
    weighted = X * powers_inv
    global_cum = np.cumsum(weighted)
    
    # Векторизованная сегментированная кумулятивная сумма
    # 1. Создаем маску сброса на началах сегментов (кроме первого)
    resets = indices[1:-1]
    mask = np.zeros(I, dtype=int)
    if len(resets) > 0:
        mask[resets] = resets
        
    # 2. Для каждой позиции находим индекс последнего сброса
    last_reset = np.maximum.accumulate(mask)
    
    # 3. Вычисляем величину коррекции (значение глобальной суммы перед сбросом)
    correction = np.zeros(I)
    valid = last_reset > 0
    correction[valid] = global_cum[last_reset[valid] - 1]
    
    # 4. Сегментированная сумма
    seg_cum = global_cum - correction
    
    # Финальный расчет EMA
    Y = (1.0 - gamma) * powers * seg_cum
    return Y

In [10]:
X = [10.] * 20
indices = [0, 3, 10, 20]
gamma = 0.6

In [11]:
attn(X, indices, gamma)

array([4.        , 6.4       , 7.84      , 4.        , 6.4       ,
       7.84      , 8.704     , 9.2224    , 9.53344   , 9.720064  ,
       4.        , 6.4       , 7.84      , 8.704     , 9.2224    ,
       9.53344   , 9.720064  , 9.8320384 , 9.89922304, 9.93953382])

In [12]:
attn_vectorized(X, indices, gamma)

array([4.        , 6.4       , 7.84      , 4.        , 6.4       ,
       7.84      , 8.704     , 9.2224    , 9.53344   , 9.720064  ,
       4.        , 6.4       , 7.84      , 8.704     , 9.2224    ,
       9.53344   , 9.720064  , 9.8320384 , 9.89922304, 9.93953382])